In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

2025-03-10 10:49:41.847107: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-10 10:49:42.743859: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-10 10:49:42.743914: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-10 10:49:42.862603: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-10 10:49:43.124843: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the models required
frozen_model = tf.keras.models.load_model('models/ResNet_CNN_LSTM_frozen.keras')
finetuned_model = tf.keras.models.load_model('models/ResNet_CNN_LSTM_finetuned.keras')

2025-03-10 10:50:10.202763: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_count']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

Block 0103

Predictions wth the frozen model

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-10 10:56:20.380069: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 203s 504ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'frozen_block_0103_ResNet')

/tmp/ipykernel_4096730/2756840855.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))


In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

[35.714285714285715,
 36.08719598813011,
 PearsonRResult(statistic=nan, pvalue=nan),
 -47.63719512195122]

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,0.0
1,Block0103_2020_08_27,39,39.000001,0.0
2,Block0103_2020_08_28,41,41.000000,0.0
3,Block0103_2020_08_31,31,31.000000,0.0
4,Block0103_2020_09_02,32,32.000000,0.0
5,Block0103_2020_09_07,40,40.002086,0.0
6,Block0103_2020_09_16,27,27.000176,0.0


Predictions with the finetuned model

In [14]:
%%time
# first get the predictions
finetuned_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', finetuned_model)

384/384 [==============================] - 197s 508ms/step
CPU times: user 3min 35s, sys: 4.87 s, total: 3min 40s
Wall time: 3min 25s


In [15]:
finetuned_preds_block_0103.shape

(12288, 7)

In [16]:
finetuned_final_forecasts_block_0103 = get_final_forecasted_and_true_values(finetuned_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'finetuned_block_0103_ResNet')

/tmp/ipykernel_4096730/2756840855.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))


In [17]:
finetuned_normalized_forecasts_block_0103 = finetuned_final_forecasts_block_0103[0]

In [18]:
print(finetuned_normalized_forecasts_block_0103)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [19]:
mae_finetuned_block_0103 = finetuned_final_forecasts_block_0103[1]
mae_finetuned_block_0103

[35.714285714285715,
 36.08719598813011,
 PearsonRResult(statistic=nan, pvalue=nan),
 -47.63719512195122]

In [20]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0103[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,0.0
1,Block0103_2020_08_27,39,39.000001,0.0
2,Block0103_2020_08_28,41,41.000000,0.0
3,Block0103_2020_08_31,31,31.000000,0.0
4,Block0103_2020_09_02,32,32.000000,0.0
5,Block0103_2020_09_07,40,40.002086,0.0
6,Block0103_2020_09_16,27,27.000176,0.0


Block 0104

Predictions wth the frozen model

In [21]:
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 195s 508ms/step


In [22]:
frozen_preds_block_0104.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'frozen_block_0104_ResNet')

/tmp/ipykernel_4096730/2756840855.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))


In [24]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [25]:
print(frozen_normalized_forecasts_block_0104)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [26]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

[36.42857142857143,
 36.7520650537393,
 PearsonRResult(statistic=nan, pvalue=nan),
 -56.05603448275862]

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,0.0
1,Block0104_2020_08_27,30,30.000000,0.0
2,Block0104_2020_08_28,39,39.000001,0.0
3,Block0104_2020_08_31,40,40.000000,0.0
4,Block0104_2020_09_02,41,40.998810,0.0
5,Block0104_2020_09_07,42,42.169009,0.0
6,Block0104_2020_09_16,30,30.005317,0.0


Predictions with the finetuned model

In [28]:
# first get the predictions
finetuned_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', finetuned_model)

384/384 [==============================] - 195s 509ms/step


In [29]:
finetuned_preds_block_0104.shape

(12288, 7)

In [30]:
finetuned_final_forecasts_block_0104 = get_final_forecasted_and_true_values(finetuned_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'finetuned_block_0104_ResNet')

/tmp/ipykernel_4096730/2756840855.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))


In [31]:
finetuned_normalized_forecasts_block_0104 = finetuned_final_forecasts_block_0104[0]

In [32]:
print(finetuned_normalized_forecasts_block_0104)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [33]:
mae_finetuned_block_0104 = finetuned_final_forecasts_block_0104[1]
mae_finetuned_block_0104

[36.42857142857143,
 36.7520650537393,
 PearsonRResult(statistic=nan, pvalue=nan),
 -56.05603448275862]

In [34]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0104[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,0.0
1,Block0104_2020_08_27,30,30.000000,0.0
2,Block0104_2020_08_28,39,39.000001,0.0
3,Block0104_2020_08_31,40,40.000000,0.0
4,Block0104_2020_09_02,41,40.998810,0.0
5,Block0104_2020_09_07,42,42.169009,0.0
6,Block0104_2020_09_16,30,30.005317,0.0


Block 0105

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0105.shape

In [ ]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'frozen_block_0105_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [ ]:
print(frozen_normalized_forecasts_block_0105)

In [ ]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0105.shape

In [ ]:
finetuned_final_forecasts_block_0105 = get_final_forecasted_and_true_values(finetuned_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'finetuned_block_0105_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0105 = finetuned_final_forecasts_block_0105[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0105)

In [ ]:
mae_finetuned_block_0105 = finetuned_final_forecasts_block_0105[1]
mae_finetuned_block_0105

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0105[2]
finetuned_true_forecasted_df

Block 0106

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0106.shape

In [ ]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'frozen_block_0106_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [ ]:
print(frozen_normalized_forecasts_block_0106)

In [ ]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0106.shape

In [ ]:
finetuned_final_forecasts_block_0106 = get_final_forecasted_and_true_values(finetuned_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'finetuned_block_0106_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0106 = finetuned_final_forecasts_block_0106[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0106)

In [ ]:
mae_finetuned_block_0106 = finetuned_final_forecasts_block_0106[1]
mae_finetuned_block_0106

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0106[2]
finetuned_true_forecasted_df

Block 0201

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0201.shape

In [ ]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'frozen_block_0201_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [ ]:
print(frozen_normalized_forecasts_block_0201)

In [ ]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0201.shape

In [ ]:
finetuned_final_forecasts_block_0201 = get_final_forecasted_and_true_values(finetuned_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'finetuned_block_0201_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0201 = finetuned_final_forecasts_block_0201[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0201)

In [ ]:
mae_finetuned_block_0201 = finetuned_final_forecasts_block_0201[1]
mae_finetuned_block_0201

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0201[2]
finetuned_true_forecasted_df

Block 0202

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0202.shape

In [ ]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'frozen_block_0202_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [ ]:
print(frozen_normalized_forecasts_block_0202)

In [ ]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0202.shape

In [ ]:
finetuned_final_forecasts_block_0202 = get_final_forecasted_and_true_values(finetuned_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'finetuned_block_0202_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0202 = finetuned_final_forecasts_block_0202[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0202)

In [ ]:
mae_finetuned_block_0202 = finetuned_final_forecasts_block_0202[1]
mae_finetuned_block_0202

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0202[2]
finetuned_true_forecasted_df

Block 0205

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0205.shape

In [ ]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'frozen_block_0205_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [ ]:
print(frozen_normalized_forecasts_block_0205)

In [ ]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0205.shape

In [ ]:
finetuned_final_forecasts_block_0205 = get_final_forecasted_and_true_values(finetuned_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'finetuned_block_0205_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0205 = finetuned_final_forecasts_block_0205[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0205)

In [ ]:
mae_finetuned_block_0205 = finetuned_final_forecasts_block_0205[1]
mae_finetuned_block_0205

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0205[2]
finetuned_true_forecasted_df

Block 0206

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0206.shape

In [ ]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'frozen_block_0206_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [ ]:
print(frozen_normalized_forecasts_block_0206)

In [ ]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0206.shape

In [ ]:
finetuned_final_forecasts_block_0206 = get_final_forecasted_and_true_values(finetuned_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'finetuned_block_0206_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0206 = finetuned_final_forecasts_block_0206[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0206)

In [ ]:
mae_finetuned_block_0206 = finetuned_final_forecasts_block_0206[1]
mae_finetuned_block_0206

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0206[2]
finetuned_true_forecasted_df

Block 0302

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0302.shape

In [ ]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'frozen_block_0302_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [ ]:
print(frozen_normalized_forecasts_block_0302)

In [ ]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0302.shape

In [ ]:
finetuned_final_forecasts_block_0302 = get_final_forecasted_and_true_values(finetuned_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'finetuned_block_0302_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0302 = finetuned_final_forecasts_block_0302[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0302)

In [ ]:
mae_finetuned_block_0302 = finetuned_final_forecasts_block_0302[1]
mae_finetuned_block_0302

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0302[2]
finetuned_true_forecasted_df

Block 0303

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0303.shape

In [ ]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'frozen_block_0303_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [ ]:
print(frozen_normalized_forecasts_block_0303)

In [ ]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0303.shape

In [ ]:
finetuned_final_forecasts_block_0303 = get_final_forecasted_and_true_values(finetuned_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'finetuned_block_0303_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0303 = finetuned_final_forecasts_block_0303[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0303)

In [ ]:
mae_finetuned_block_0303 = finetuned_final_forecasts_block_0303[1]
mae_finetuned_block_0303

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0303[2]
finetuned_true_forecasted_df

Block 0304

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0304.shape

In [ ]:
frozen_final_forecasts_block_0304 = get_final_forecasted_and_true_values(frozen_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'frozen_block_0304_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0304 = frozen_final_forecasts_block_0304[0]

In [ ]:
print(frozen_normalized_forecasts_block_0304)

In [ ]:
mae_frozen_block_0304 = frozen_final_forecasts_block_0304[1]
mae_frozen_block_0304

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0304[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0304.shape

In [ ]:
finetuned_final_forecasts_block_0304 = get_final_forecasted_and_true_values(finetuned_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'finetuned_block_0304_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0304 = finetuned_final_forecasts_block_0304[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0304)

In [ ]:
mae_finetuned_block_0304 = finetuned_final_forecasts_block_0304[1]
mae_finetuned_block_0304

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0304[2]
finetuned_true_forecasted_df

Block 0305

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0305.shape

In [ ]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'frozen_block_0305_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [ ]:
print(frozen_normalized_forecasts_block_0305)

In [ ]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0305.shape

In [ ]:
finetuned_final_forecasts_block_0305 = get_final_forecasted_and_true_values(finetuned_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'finetuned_block_0305_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0305 = finetuned_final_forecasts_block_0305[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0305)

In [ ]:
mae_finetuned_block_0305 = finetuned_final_forecasts_block_0305[1]
mae_finetuned_block_0305

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0305[2]
finetuned_true_forecasted_df

Block 0306

Predictions wth the frozen model

In [ ]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

In [ ]:
frozen_preds_block_0306.shape

In [ ]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'frozen_block_0306_ResNet')

In [ ]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [ ]:
print(frozen_normalized_forecasts_block_0306)

In [ ]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

In [ ]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

Predictions with the finetuned model

In [ ]:
# first get the predictions
finetuned_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', finetuned_model)

In [ ]:
finetuned_preds_block_0306.shape

In [ ]:
finetuned_final_forecasts_block_0306 = get_final_forecasted_and_true_values(finetuned_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'finetuned_block_0306_ResNet')

In [ ]:
finetuned_normalized_forecasts_block_0306 = finetuned_final_forecasts_block_0306[0]

In [ ]:
print(finetuned_normalized_forecasts_block_0306)

In [ ]:
mae_finetuned_block_0306 = finetuned_final_forecasts_block_0306[1]
mae_finetuned_block_0306

In [ ]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0306[2]
finetuned_true_forecasted_df